In [1]:
import pandas as pd
import mlcroissant as mlc
import hashlib

In [2]:
def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

In [3]:
def field(record_set, column, dtype, description, file_id, separator=None):
    return mlc.Field(
        id = f"{record_set}/{column}",
        name = column,
        description = description,
        data_types = dtype,
        source=mlc.Source(
            file_object=file_id,
            extract=mlc.Extract(column=column),
            transforms=[mlc.Transform(separator=separator)] if separator else [],
		),
        repeated=bool(separator),
	)

In [4]:
# !uv pip install mlcroissant

In [5]:
base_path = "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data_with_repair"
stream_path = f"{base_path}/wind_turbine_stream_data_with_repair.parquet"
events_path = f"{base_path}/wind_turbine_fault_events_with_repair.parquet"

In [6]:
stream_hash = sha256(stream_path)
events_hash = sha256(events_path)

In [7]:
stream_hash

'213d9e5f59de3dc9b0e3a24417a673c070015d6a259ab1b1e3aaaab1856d136a'

In [8]:
events_hash

'684ab212ebc12837148f7fb4075ca3821f0b6356153c9572a8f2e0f03a9eab67'

In [9]:
# for i,val in enumerate(stream_hash):
#     if val == "4":
#         print(i)
# for i,val in enumerate(events_hash):
#     if val == "4":
#         print(i)

In [10]:
stream_file = mlc.FileObject(
    id = "wind_with_repair_stream.parquet",
    name = "wind_with_repair_stream.parquet",
    description="Per-turbine sensor data at 10min agg.",
    content_url = stream_path,
    encoding_formats=["application/x-parquet"],
    sha256=stream_hash
)

stream_file = mlc.FileObject(
    id = "wind_with_repair_event.parquet",
    name = "wind_with_repair_event.parquet",
    description="Per-turbine events data at 10min agg.",
    content_url = events_path,
    encoding_formats=["application/x-parquet"],
    sha256=events_hash
)

In [11]:
stream_file

FileObject(uuid="wind_with_repair_event.parquet")

In [12]:
df = pd.read_parquet(stream_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525600 entries, 0 to 525599
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   time                      525600 non-null  datetime64[ns]
 1   device                    525600 non-null  object        
 2   active_power              525600 non-null  float64       
 3   wind_speed                525600 non-null  float64       
 4   air_density               525600 non-null  float64       
 5   wind_direction            525600 non-null  float64       
 6   nacelle_direction         525600 non-null  float64       
 7   nacelle_position          525600 non-null  float64       
 8   ambient_temp              525600 non-null  float64       
 9   rotor_speed               525600 non-null  float64       
 10  generator_speed           525600 non-null  float64       
 11  gearbox_oil_temp          525600 non-null  float64       
 12  ge

In [13]:
df["fault_temperature"].unique()

array([0, 1])

In [14]:
df["fault_severity"].unique()

array(['none', 'low', 'medium', 'high'], dtype=object)

In [15]:
df["fault_labels"].unique()

array(['healthy', 'temperature', 'yaw_misalignment', 'pitch_misalignment',
       'temperature|yaw_misalignment'], dtype=object)

In [16]:
sensors_float = [x for x in df.select_dtypes([float,int]).columns if ((not x.startswith("fault_")) and x != "is_drifted")]
sensors_float

['active_power',
 'wind_speed',
 'air_density',
 'wind_direction',
 'nacelle_direction',
 'nacelle_position',
 'ambient_temp',
 'rotor_speed',
 'generator_speed',
 'gearbox_oil_temp',
 'generator_temp',
 'bearing_temp',
 'converter_temp',
 'pitch_blade_angle_1',
 'pitch_blade_angle_2',
 'pitch_blade_angle_3']

In [17]:
sensors_with_desc = [
    ("active_power", "Active power output in kW."),
    ("wind_speed", "Wind speed at the turbine in m/s."),
    ("air_density", "Air density in kg/m^3."),
    ("wind_direction", "Wind direction in degrees."),
    ("nacelle_direction", "Nacelle heading in degrees."),
    ("nacelle_position", "Nacelle position in degrees."),
    ("ambient_temp", "Ambient temperature in Celsius."),
    ("rotor_speed", "Rotor speed in rpm."),
    ("generator_speed", "Generator speed in rpm."),
    ("gearbox_oil_temp", "Gearbox oil temperature in Celsius."),
    ("generator_temp", "Generator temperature in Celsius."),
    ("bearing_temp", "Bearing temperature in Celsius."),
    ("converter_temp", "Converter temperature in Celsius."),
    ("pitch_blade_angle_1", "Pitch angle of blade 1 in degrees."),
    ("pitch_blade_angle_2", "Pitch angle of blade 2 in degrees."),
    ("pitch_blade_angle_3", "Pitch angle of blade 3 in degrees."),
]

In [18]:
# fault_flags = [x for x in df.select_dtypes([float,int, object]).columns if ((x.startswith("fault_")) or x == "is_drifted")]


fault_flags = [
    ("fault_temperature", "temperature"),
    ("fault_pitch_misalignment", "pitch misalignment"),
    ("fault_yaw_misalignment", "yaw misalignment"),
]


In [19]:
stream_id = "wind_with_repair_stream.parquet"

In [20]:
measurements = mlc.RecordSet(
    id="measurements",
    name="measurements",
    description="One record per turbine per 10 minutes. Ground Truth is in 3 forms: is_drifted meaning any fault, fault_labels meaning which fault which is multivalued separated by |, and 3 binary fault_* columns which are one hot encoded of fault_labels.",
    fields=[
        field("measurements", "time", mlc.DataType.DATE, "Timestamp of the reading.", stream_id),
        field("measurements", "device", mlc.DataType.TEXT, "Individual turbine name.", stream_id),
        *[field("measurements", col, mlc.DataType.FLOAT, desc, stream_id) for col, desc in sensors_with_desc],
        field("measurements", "fault_labels", mlc.DataType.TEXT, "Active fault, multivalued, pipe separated. Primary multi label target.", stream_id),
        field("measurements", "fault_severity", mlc.DataType.TEXT, "Severity of the active fault. One of: none, low, medium, high.", stream_id),
        field("measurements", "drift_start_time", mlc.DataType.DATE, "Timestamp When drift started.", stream_id),
        *[field("measurements", c, mlc.DataType.INTEGER,
                f"1 if a {label} fault is active at this step, else 0. One-hot form of fault_labels.",
                stream_id)
          for c, label in fault_flags],
	]
)

In [21]:
dfx = pd.read_parquet(events_path)
dfx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   device          10 non-null     object        
 1   fault_type      10 non-null     object        
 2   start_time      10 non-null     datetime64[ns]
 3   end_time        5 non-null      datetime64[ns]
 4   shape           10 non-null     object        
 5   max_severity    10 non-null     float64       
 6   ramp_steps      1 non-null      float64       
 7   ramp_days       8 non-null      float64       
 8   ramp_down_days  2 non-null      float64       
dtypes: datetime64[ns](2), float64(4), object(3)
memory usage: 852.0+ bytes


In [23]:
for col in df.columns:
    print(col)
    print(df[col].unique())

time
<DatetimeArray>
['2025-01-01 00:00:00', '2025-01-01 00:10:00', '2025-01-01 00:20:00',
 '2025-01-01 00:30:00', '2025-01-01 00:40:00', '2025-01-01 00:50:00',
 '2025-01-01 01:00:00', '2025-01-01 01:10:00', '2025-01-01 01:20:00',
 '2025-01-01 01:30:00',
 ...
 '2025-12-31 22:20:00', '2025-12-31 22:30:00', '2025-12-31 22:40:00',
 '2025-12-31 22:50:00', '2025-12-31 23:00:00', '2025-12-31 23:10:00',
 '2025-12-31 23:20:00', '2025-12-31 23:30:00', '2025-12-31 23:40:00',
 '2025-12-31 23:50:00']
Length: 52560, dtype: datetime64[ns]
device
['WT001' 'WT002' 'WT003' 'WT004' 'WT005' 'WT006' 'WT007' 'WT008' 'WT009'
 'WT010']
active_power
[ 950.56 1048.92  919.58 ... 1654.99 1797.74 1703.45]
wind_speed
[7.84 8.05 7.71 ... 1.35 1.17 1.13]
air_density
[1.2852 1.2837 1.2802 ... 1.3281 1.3248 1.3251]
wind_direction
[221.11 219.31 220.85 ... 156.51 164.12 159.81]
nacelle_direction
[223.03 219.77 221.27 ... 163.93 165.19 165.45]
nacelle_position
[223.47 219.62 221.26 ... 167.51 166.73 165.15]
ambient_tem